In [24]:
# === BLOQUE 0 · LIBRERÍAS Y CONFIG ===
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


1) Columnas canónicas y archivo de entrada

In [25]:
RUTA = r"C:\Users\Vìctor\OneDrive\Desktop\ESP_Project"
FILE = "df_con_eventos_hibridos.xlsx"   # tu dataset actual

df = pd.read_excel(Path(RUTA)/FILE)
print(">> Columnas originales:")
print(df.columns.tolist())

>> Columnas originales:
['name_', 'date', 'prueba_de_produccion_petróleo_a_24_horas_bbld', 'prueba_de_producción_agua_a_24_horas_bbld', 'prueba_pozooil_24__prueba_pozowater_24', 'prueba_de_producción_gas_a_24_horas_mcfd', 'bsw_pct', 'gravedad_api_del_petroleo', 'salinidad_ppm_pu', 'presion_de_intake_psi', 'frecuencia_bomba_hz', 'amperaje_bomba_amp', 'presion_de_tubing_psi', 'presion_de_casing_psi', 'tipo_de_bomba', 'temperatura_de_la_bomba_deg_f', 'etapas_de_la_bomba', 'campo', 'delta_1', 'delta_3', 'slope_7', 'slope_14', 'evento_hibrido', 'Unnamed: 23']


1) Configuración y helpers

In [ ]:
# --- Columnas de tu dataset (no cambiamos nombres) ---
COL_POZO   = "name_"
COL_FECHA  = "date"
COL_QTOT   = "prueba_pozooil_24__prueba_pozowater_24"
COL_QGAS   = "prueba_de_producción_gas_a_24_horas_mcfd"
COL_HZ     = "frecuencia_bomba_hz"
COL_PINT   = "presion_de_intake_psi"
COL_BOMBA  = "tipo_de_bomba"
COL_EVENTO = "evento_hibrido"    # (0 normal, 1 caída, 2 reacond.)

# Si no existe, generamos un "régimen" (pozo||bomba normalizada)
if "regimen_id" not in df.columns:
    bomba_norm = (df[COL_BOMBA].astype(str)
                             .str.strip()
                             .replace({"": np.nan, "nan": np.nan}))
    df["regimen_id"] = df[COL_POZO].astype(str) + "||" + bomba_norm.fillna("NA")
COL_REG = "regimen_id"

# Orden temporal seguro
df[COL_FECHA] = pd.to_datetime(df[COL_FECHA], errors="coerce")
df = df.sort_values([COL_POZO, COL_FECHA], na_position="last").reset_index(drop=True)

# Umbrales (operacionales y confirmación)
Q_LOW, Q_HIGH = 0.30, 0.70          # sobre operativo (más estrecho)
Q_GAS_HI      = 0.80                # gate gas (más sensible para gas)
Q_HZ_HI       = 0.90                # apoyo: Hz alto
Q_PINT_LO     = 0.10                # apoyo: Pint bajo

-K_SLOPE       = 1.0                 # |slope_7 - μ| > k·σ
P_D1          = 80                  # percentil de |Δ1| por pozo (shock)
RATIO_UP      = 0.20                # salto relativo para subidas (opcional)

N_PERSIST     = 2                   # racha mínima
K_REFRACT     = 1                   # silencio tras disparo

# Helpers
def _q(s, q):
    s = pd.to_numeric(s, errors="coerce").dropna()
    return np.nan if s.empty else np.nanpercentile(s, q*100)

def persistent_flag_array(x, n):
    """Marca 1 cuando la suma en ventana n (consecutivos) >= n."""
    # x: vector 0/1
    if len(x) == 0:
        return x
    # Rolling suma: 1 si hay n consecutivos
    roll = pd.Series(x).rolling(window=n, min_periods=n).sum().fillna(0).to_numpy()
    return (roll >= n).astype(int)

def refractory_filter_series(s, k):
    """Silencia k lecturas después de cada 1."""
    y = s.to_numpy().copy()
    cool = 0
    for i in range(len(y)):
        if cool > 0:
            y[i] = 0
            cool -= 1
        elif y[i] == 1:
            cool = k
    return pd.Series(y, index=s.index, dtype=int)

2) Sobre operativo por pozo×régimen con fallback a pozo

In [27]:
# a) Percentiles por pozo×régimen (cuando hay suficientes filas)
MIN_FILAS_REG = 8  # si un régimen tiene menos de esto, cae a sobre por pozo

sizes_reg = df.groupby([COL_POZO, COL_REG]).size()
reg_enough = sizes_reg[sizes_reg >= MIN_FILAS_REG].index

q_reg = (df.set_index([COL_POZO, COL_REG])
           .loc[reg_enough]
           .groupby(level=[0,1])
           .agg(
               qtot_lo = (COL_QTOT,  lambda s: _q(s, Q_LOW)),
               qtot_hi = (COL_QTOT,  lambda s: _q(s, Q_HIGH)),
               gas_hi  = (COL_QGAS,  lambda s: _q(s, Q_GAS_HI)),
               hz_hi   = (COL_HZ,    lambda s: _q(s, Q_HZ_HI)),
               pint_lo = (COL_PINT,  lambda s: _q(s, Q_PINT_LO)),
           ))

# b) Percentiles por pozo (fallback)
q_pozo = (df.groupby(COL_POZO).agg(
            qtot_lo_p = (COL_QTOT,  lambda s: _q(s, Q_LOW)),
            qtot_hi_p = (COL_QTOT,  lambda s: _q(s, Q_HIGH)),
            gas_hi_p  = (COL_QGAS,  lambda s: _q(s, Q_GAS_HI)),
            hz_hi_p   = (COL_HZ,    lambda s: _q(s, Q_HZ_HI)),
            pint_lo_p = (COL_PINT,  lambda s: _q(s, Q_PINT_LO)),
         ))

# c) Unir thresholds al dataframe
df2 = df.merge(q_reg, left_on=[COL_POZO, COL_REG], right_index=True, how="left") \
        .merge(q_pozo, left_on=COL_POZO, right_index=True, how="left")

# d) Efectivo = régimen si existe, si no pozo
for base, fallback in [("qtot_lo","qtot_lo_p"), ("qtot_hi","qtot_hi_p"),
                       ("gas_hi","gas_hi_p"), ("hz_hi","hz_hi_p"),
                       ("pint_lo","pint_lo_p")]:
    df2[f"{base}_eff"] = np.where(df2[base].notna(), df2[base], df2[fallback])


3) Señales operativas (env_q y gate gas)

In [28]:
# 3.1 Dentro del sobre de Q_total
def flag_env_q(row):
    q, lo, hi = row[COL_QTOT], row["qtot_lo_eff"], row["qtot_hi_eff"]
    if pd.isna(q) or pd.isna(lo) or pd.isna(hi):
        return np.nan
    return 1 if (lo <= q <= hi) else 0

df2["env_q"] = df2.apply(flag_env_q, axis=1)

# 3.2 Gate por gas + condición operativa (Hz alto o Pint bajo)
def flag_gate(row):
    g, gh = row[COL_QGAS], row["gas_hi_eff"]
    hz, hh = row[COL_HZ],  row["hz_hi_eff"]
    pi, pl = row[COL_PINT], row["pint_lo_eff"]

    if pd.isna(g) or pd.isna(gh):
        return 0
    gas_out = (g > gh)

    hz_out = (not pd.isna(hz)) and (not pd.isna(hh)) and (hz > hh)
    pi_out = (not pd.isna(pi)) and (not pd.isna(pl)) and (pi < pl)

    return int(gas_out and (hz_out or pi_out))

df2["env_gate"] = df2.apply(flag_gate, axis=1)


4) Confirmación asimétrica (caída vs subida) usando lo que YA tienes (slope_7, delta_1)

In [30]:
# Estadística por pozo para slope y |Δ1|
stats_pozo = (df2.groupby(COL_POZO)
                .agg(slope7_mean=("slope_7","mean"),
                     slope7_std =("slope_7","std"),
                     d1_p       =("delta_1", lambda s: _q(np.abs(s), P_D1/100)))
             )

df2 = df2.merge(stats_pozo, left_on=COL_POZO, right_index=True, how="left")

# Opcional: ratio relativo vs 14 lecturas atrás (si no te convence, puedes omitirlo)
VOL_EPS = 1e-3  # evita divisiones por ~0 en ratio14
df2["Qt_prev14"] = (df2.groupby(COL_POZO)[COL_QTOT].shift(14))
df2["ratio14"]   = np.where(
    df2["Qt_prev14"].abs() > VOL_EPS,
    (df2[COL_QTOT] - df2["Qt_prev14"]) / df2["Qt_prev14"],
    np.nan
)

def confirm_asimetrica(row):
    s7, mu, sd = row["slope_7"], row["slope7_mean"], row["slope7_std"]
    d1, d1p    = row["delta_1"], row["d1_p"]
    r14        = row["ratio14"]

    ok = False

    # Caída fuerte
    if (not pd.isna(s7) and not pd.isna(mu) and not pd.isna(sd) and (s7 - mu) < -K_SLOPE*sd) \
       or (not pd.isna(d1) and not pd.isna(d1p) and d1 < -d1p):
        ok = True

    # Subida/reacondicionamiento
    if (not pd.isna(s7) and not pd.isna(mu) and not pd.isna(sd) and (s7 - mu) >  K_SLOPE*sd) \
       or (not pd.isna(d1) and not pd.isna(d1p) and d1 >  d1p) \
       or (not pd.isna(r14) and r14 > RATIO_UP):
        ok = True

    return int(ok)

df2["confirm"] = df2.apply(confirm_asimetrica, axis=1)


5) Alarma instantánea + persistencia + refractario

In [31]:
# Señal mínima: al menos 2 de las 4 variables relevantes presentes
def enough_signals(row):
    signals = [row[COL_QTOT], row[COL_QGAS], row[COL_HZ], row[COL_PINT]]
    return int(sum(pd.notna(signals)) >= 2)

df2["enough"] = df2.apply(enough_signals, axis=1)

# Fuera de sobre instantáneo si (Q_total fuera) o (gate gas ON)
df2["fuera_inst"] = np.where((df2["env_q"]==0) | (df2["env_gate"]==1), 1, 0)

# Alarma operativa instantánea (con confirmación y señales suficientes)
df2["alarma_instantanea"] = np.where(
    (df2["fuera_inst"]==1) & (df2["confirm"]==1) & (df2["enough"]==1), 1, 0
).astype(int)

# Persistencia: racha de N consecutivas (rolling)
df2["persist"] = (df2.groupby(COL_POZO, group_keys=False)["alarma_instantanea"]
                    .apply(lambda s: pd.Series(
                        persistent_flag_array(s.fillna(0).astype(int).to_numpy(), N_PERSIST),
                        index=s.index))
                 ).astype(int)

# Refractario: silencia K lecturas tras cada 1 persistente
df2["persist_ref"] = (df2.groupby(COL_POZO, group_keys=False)["persist"]
                        .apply(lambda s: refractory_filter_series(s, K_REFRACT))
                     ).astype(int)


In [32]:
import numpy as np
import pandas as pd

COL_EVENTO = "evento_hibrido"

def binariza_evento(s):
    """0 = normal; 1 = (1 ó 2) anomalía."""
    return (s.fillna(0).astype(int) > 0).astype(int)

def confusion_binaria(y_true_bin, y_pred_bin):
    """Devuelve TP, FP, FN, TN en binario."""
    tp = int(((y_true_bin == 1) & (y_pred_bin == 1)).sum())
    fp = int(((y_true_bin == 0) & (y_pred_bin == 1)).sum())
    fn = int(((y_true_bin == 1) & (y_pred_bin == 0)).sum())
    tn = int(((y_true_bin == 0) & (y_pred_bin == 0)).sum())
    return tp, fp, fn, tn

def metricas(tp, fp, fn, tn):
    prec = tp / (tp+fp) if (tp+fp)>0 else 0.0
    rec  = tp / (tp+fn) if (tp+fn)>0 else 0.0
    spec = tn / (tn+fp) if (tn+fp)>0 else 0.0
    bacc = 0.5*(rec+spec)
    f1   = (2*prec*rec)/(prec+rec) if (prec+rec)>0 else 0.0
    return prec, rec, f1, bacc

# 1) Vector binario “verdad”
y_true = binariza_evento(df2[COL_EVENTO])

# 2) Evalúa diferentes salidas del detector
for nombre_pred in ["alarma_instantanea", "persist", "persist_ref"]:
    if nombre_pred not in df2.columns:
        continue
    y_pred = df2[nombre_pred].fillna(0).astype(int)
    tp, fp, fn, tn = confusion_binaria(y_true, y_pred)
    prec, rec, f1, bacc = metricas(tp, fp, fn, tn)

    print(f"\n>>> Evaluación: {nombre_pred}")
    print(f"TP={tp}  FP={fp}  FN={fn}  TN={tn}")
    print(f"Precision={prec:.3f}  Recall={rec:.3f}  F1={f1:.3f}  BalancedAcc={bacc:.3f}")

    # Tabla 2x2 para revisar
    print(pd.crosstab(y_true, y_pred, rownames=["y_true"], colnames=[nombre_pred]))



>>> Evaluación: alarma_instantanea
TP=2479  FP=1171  FN=1290  TN=11221
Precision=0.679  Recall=0.658  F1=0.668  BalancedAcc=0.782
alarma_instantanea      0     1
y_true                         
0                   11221  1171
1                    1290  2479

>>> Evaluación: persist
TP=1218  FP=660  FN=2551  TN=11732
Precision=0.649  Recall=0.323  F1=0.431  BalancedAcc=0.635
persist      0     1
y_true              
0        11732   660
1         2551  1218

>>> Evaluación: persist_ref
TP=795  FP=402  FN=2974  TN=11990
Precision=0.664  Recall=0.211  F1=0.320  BalancedAcc=0.589
persist_ref      0    1
y_true                 
0            11990  402
1             2974  795


6) Chequeo rápido y métricas (si tienes evento_hibrido)

In [33]:
# Cobertura de thresholds
print("Cobertura thresholds efectivos:")
print(df2[["qtot_lo_eff","qtot_hi_eff","gas_hi_eff","hz_hi_eff","pint_lo_eff"]].notna().mean())

# Conteos básicos
print("\nValue counts env_q / env_gate / confirm:")
print("env_q:\n", df2["env_q"].value_counts(dropna=False))
print("env_gate:\n", df2["env_gate"].value_counts(dropna=False))
print("confirm:\n", df2["confirm"].value_counts(dropna=False))

# Métricas vs tu etiqueta (si existe)
if COL_EVENTO in df2.columns:
    def resumen(tab, pos_label=1):
        TP = tab.get((pos_label,1), 0)
        FP = tab.get((0,1), 0)
        FN = tab.get((pos_label,0), 0)
        TN = tab.get((0,0), 0)
        prec  = TP / (TP+FP) if (TP+FP)>0 else 0.0
        rec   = TP / (TP+FN) if (TP+FN)>0 else 0.0
        spec  = TN / (TN+FP) if (TN+FP)>0 else 0.0
        bacc  = 0.5*(rec+spec)
        return prec, rec, bacc

    print("\n>> Tabla cruzada: evento_hibrido vs alarma_instantanea")
    c_inst = pd.crosstab(df2[COL_EVENTO], df2["alarma_instantanea"])
    print(c_inst)
    p,r,b = resumen(c_inst)
    print(f"Precision={p:.3f}  Recall={r:.3f}  BalancedAcc={b:.3f}")

    print("\n>> Persistente (≥N):")
    c_p = pd.crosstab(df2[COL_EVENTO], df2["persist"])
    print(c_p)
    p,r,b = resumen(c_p)
    print(f"Precision={p:.3f}  Recall={r:.3f}  BalancedAcc={b:.3f}")

    print("\n>> Persistente + Refractario:")
    c_pr = pd.crosstab(df2[COL_EVENTO], df2["persist_ref"])
    print(c_pr)
    p,r,b = resumen(c_pr)
    print(f"Precision={p:.3f}  Recall={r:.3f}  BalancedAcc={b:.3f}")


Cobertura thresholds efectivos:
qtot_lo_eff   1.00
qtot_hi_eff   1.00
gas_hi_eff    1.00
hz_hi_eff     1.00
pint_lo_eff   1.00
dtype: float64

Value counts env_q / env_gate / confirm:
env_q:
 env_q
0    9049
1    7112
Name: count, dtype: int64
env_gate:
 env_gate
0    15870
1      291
Name: count, dtype: int64
confirm:
 confirm
0    10518
1     5643
Name: count, dtype: int64

>> Tabla cruzada: evento_hibrido vs alarma_instantanea
alarma_instantanea      0     1
evento_hibrido                 
0                   11221  1171
1                     618  1266
2                     672  1213
Precision=0.000  Recall=0.000  BalancedAcc=0.000

>> Persistente (≥N):
persist             0    1
evento_hibrido            
0               11732  660
1                1287  597
2                1264  621
Precision=0.000  Recall=0.000  BalancedAcc=0.000

>> Persistente + Refractario:
persist_ref         0    1
evento_hibrido            
0               11990  402
1                1491  393
2           

### 6bis) Evaluación 0/1/2 (normal / decaimiento / incremento)


Usamos tus etiquetas `evento_hibrido` (0 normal, 1 caída, 2 incremento) y derivamos un `pred_evento` a partir de la alarma y la dirección de `slope_7`/`delta_1`/`ratio14`. Se calculan matriz de confusión, métricas por clase (precisión, recall, F1), macro/weighted F1 y un resumen por pozo para ver qué pozos aportan más error.

In [34]:
import numpy as np
import pandas as pd


# Etiquetas verdad (0 normal, 1 caída, 2 incremento)
y_true_mc = df2.get(COL_EVENTO, pd.Series(dtype=int)).fillna(0).astype(int).clip(0, 2)



# Asegurar que VOL_EPS esté disponible en esta celda (definido antes, pero reaseguramos)
if 'VOL_EPS' not in globals():
    VOL_EPS = 1e-3


def _direction_from_signs(row):
    """Devuelve 1 (caída) o 2 (incremento) usando votos separados y ratio con guardas."""
    s7, d1, r14 = row.get("slope_7"), row.get("delta_1"), row.get("ratio14")
    qt_prev = row.get("Qt_prev14")
    # Solo usamos ratio si hay volumen previo suficiente
    ratio = r14 if (not pd.isna(r14) and (pd.isna(qt_prev) or abs(qt_prev) > VOL_EPS)) else np.nan
    down_votes = 0
    up_votes = 0
    for v in (s7, d1):
        if pd.isna(v):
            continue
        if v < 0:
            down_votes += 1
        elif v > 0:
            up_votes += 1
    if not pd.isna(ratio):
        if ratio < -RATIO_UP:
            down_votes += 1
        elif ratio > RATIO_UP:
            up_votes += 1
    if up_votes > down_votes:
        return 2
    if down_votes > up_votes:
        return 1
    return 1  # empate: caída conservador


def _pred_evento(row):
    if row.get("alarma_instantanea", 0) == 0:
        return 0
    return _direction_from_signs(row)



# Predicción multiclase derivada
y_pred_mc = df2.apply(_pred_evento, axis=1).astype(int)



# Matriz de confusión 3x3
mat_mc = pd.crosstab(y_true_mc, y_pred_mc, rownames=["y_true"], colnames=["y_pred"], dropna=False)



def _metrics_for_class(c):
    tp = int(((y_true_mc == c) & (y_pred_mc == c)).sum())
    fp = int(((y_true_mc != c) & (y_pred_mc == c)).sum())
    fn = int(((y_true_mc == c) & (y_pred_mc != c)).sum())
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1   = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
    support = int((y_true_mc == c).sum())
    return prec, rec, f1, support



classes = [0, 1, 2]
per_class = {c: _metrics_for_class(c) for c in classes}



macro_f1 = np.mean([per_class[c][2] for c in classes])
total = len(y_true_mc) if len(y_true_mc) > 0 else 1
weighted_f1 = sum(per_class[c][2] * per_class[c][3] for c in classes) / total
acc = float((y_true_mc == y_pred_mc).mean()) if len(y_true_mc) > 0 else 0.0



print("\nMatriz de confusión 0/1/2:")
print(mat_mc)



print("\nMétricas por clase (prec, rec, f1, soporte):")
for c in classes:
    prec, rec, f1, sup = per_class[c]
    print(f"Clase {c}: P={prec:.3f} R={rec:.3f} F1={f1:.3f} soporte={sup}")



print("\nAgregados:")
print(f"Accuracy={acc:.3f}  MacroF1={macro_f1:.3f}  WeightedF1={weighted_f1:.3f}")



# Resumen por pozo (macro F1) para detectar dónde falla más
per_pozo = []
for pozo, g in df2.groupby(COL_POZO):
    yt = g[COL_EVENTO].fillna(0).astype(int).clip(0, 2)
    yp = g.apply(_pred_evento, axis=1).astype(int)
    if len(yt) == 0:
        continue
    pc = {}
    for c in classes:
        tp = int(((yt == c) & (yp == c)).sum())
        fp = int(((yt != c) & (yp == c)).sum())
        fn = int(((yt == c) & (yp != c)).sum())
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1   = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
        pc[c] = f1
    macro = np.mean([pc[c] for c in classes])
    per_pozo.append({"pozo": pozo, "macro_f1": macro, "n": len(yt)})



per_pozo_df = pd.DataFrame(per_pozo).sort_values("macro_f1", ascending=True)

print("\nPozos ordenados por peor macro-F1 (top 10):")
print(per_pozo_df.head(10))



# Guardamos predicciones multiclase para exporte/revisión rápida
df2["pred_evento"] = y_pred_mc


Matriz de confusión 0/1/2:
y_pred      0     1    2
y_true                  
0       11221   443  728
1         618  1192   74
2         672   328  885

Métricas por clase (prec, rec, f1, soporte):
Clase 0: P=0.897 R=0.906 F1=0.901 soporte=12392
Clase 1: P=0.607 R=0.633 F1=0.620 soporte=1884
Clase 2: P=0.525 R=0.469 F1=0.496 soporte=1885

Agregados:
Accuracy=0.823  MacroF1=0.672  WeightedF1=0.821

Pozos ordenados por peor macro-F1 (top 10):
              pozo  macro_f1    n
36     SCHAR-503UI      0.46   52
35     SCHAR-500UI      0.56   87
41      SCHP-188UI      0.58  308
34     SCHAQ-497UI      0.59  106
17     SCHAB-316UI      0.59  336
23  SCHAF-539HS1UI      0.60  213
38    SCHH-237R1HS      0.60  332
15      SCHA-418UI      0.61   92
33     SCHAQ-492UI      0.61  137
37      SCHH-218HS      0.61  341

Pozos ordenados por peor macro-F1 (top 10):
              pozo  macro_f1    n
36     SCHAR-503UI      0.46   52
35     SCHAR-500UI      0.56   87
41      SCHP-188UI      0.58  308

### Snapshot del estado actual (cache) para no recalcular

Guarda `df2` ya calculado en `freeze/` para poder cargarlo directo en futuras sesiones sin rerun de las celdas anteriores.

Para cargarlo después: `df2 = pd.read_parquet(Path(RUTA)/"freeze"/"df2_snapshot_latest.parquet")`

In [38]:
from pathlib import Path
import os
import pandas as pd

CACHE_DIR = Path(RUTA) / "freeze"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

ts_cache = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
parquet_ts = CACHE_DIR / f"df2_snapshot_{ts_cache}.parquet"
parquet_latest = CACHE_DIR / "df2_snapshot_latest.parquet"

# Guardamos snapshot con timestamp y uno marcado como latest
df2.to_parquet(parquet_ts, index=False)
df2.to_parquet(parquet_latest, index=False)

print("✅ Snapshot guardado:", parquet_ts.name)
print("✅ Snapshot actualizado:", parquet_latest.name)
print("Para reusar sin recalcular: df2 = pd.read_parquet(parquet_latest)")

✅ Snapshot guardado: df2_snapshot_20251210_213902.parquet
✅ Snapshot actualizado: df2_snapshot_latest.parquet
Para reusar sin recalcular: df2 = pd.read_parquet(parquet_latest)


In [36]:
df2.head()

,name_,date,prueba_de_produccion_petróleo_a_24_horas_bbld,prueba_de_producción_agua_a_24_horas_bbld,prueba_pozooil_24__prueba_pozowater_24,prueba_de_producción_gas_a_24_horas_mcfd,bsw_pct,gravedad_api_del_petroleo,salinidad_ppm_pu,presion_de_intake_psi,...,d1_p,Qt_prev14,ratio14,confirm,enough,fuera_inst,alarma_instantanea,persist,persist_ref,pred_evento
0,SCH-007U,2015-02-02,204.37,435.73,640.10,14.00,68.07,22.40,NaN,NaN,...,104.49,NaN,NaN,0,1,1,0,0,0,0
1,SCH-007U,2015-02-09,216.90,652.63,869.53,14.00,75.05,22.40,NaN,NaN,...,104.49,NaN,NaN,1,1,1,1,0,0,2
2,SCH-007U,2015-03-05,99.29,281.49,380.78,14.00,73.92,22.40,NaN,NaN,...,104.49,NaN,NaN,1,1,1,1,1,1,1
3,SCH-007U,2015-04-05,205.33,585.15,790.48,14.00,74.02,22.40,NaN,NaN,...,104.49,NaN,NaN,1,1,1,1,1,0,1
4,SCH-007U,2015-04-07,214.01,642.02,856.03,14.00,75.00,22.40,NaN,NaN,...,104.49,NaN,NaN,1,1,1,1,1,1,2


In [37]:
import os
from pathlib import Path
from datetime import datetime

# --- Resolver carpeta base de forma robusta ---
def resolve_base():
    od = os.environ.get("OneDrive")
    if od:
        cand = Path(od) / "Desktop" / "ESP_Project"
        if cand.exists():
            return cand
    home = Path.home()
    root = home.parent
    for c in [
        home / "OneDrive" / "Desktop" / "ESP_Project",
        root / "Víctor" / "OneDrive" / "Desktop" / "ESP_Project",
        root / "Vìctor" / "OneDrive" / "Desktop" / "ESP_Project",
    ]:
        if c.exists():
            return c
    return Path.cwd()

BASE = resolve_base()
FREEZE_DIR = BASE / "freeze"
FREEZE_DIR.mkdir(parents=True, exist_ok=True)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")

# ---- 1) CSV AMIGABLE PARA EXCEL (ES-EC) ----
csv_excel = FREEZE_DIR / f"df_operativo_final_{ts}_excel.csv"

df_out = df2.copy() 
# Redondeo suave para legibilidad (ajusta si quieres)
num_cols = df_out.select_dtypes(include="number").columns
df_out[num_cols] = df_out[num_cols].round(2)

df_out.to_csv(
    csv_excel,
    index=False,
    sep=";",             # <- separador que Excel espera con decimales coma
    decimal=",",         # <- usa coma como decimal
    encoding="utf-8-sig",# <- BOM para que Excel reconozca UTF-8 y acentos
    na_rep="",           # <- dejar celdas vacías
    date_format="%Y-%m-%d"
)
print("✅ CSV para Excel guardado en:", csv_excel)

# ---- 2) (OPCIONAL) CSV TÉCNICO ESTÁNDAR ----
csv_std = FREEZE_DIR / f"df_operativo_final_{ts}.csv"
df_out.to_csv(
    csv_std,
    index=False,
    sep=",",             # estándar
    decimal=".",         # estándar
    encoding="utf-8",
    na_rep="",
    date_format="%Y-%m-%d"
)
print("✅ CSV técnico estándar guardado en:", csv_std)


✅ CSV para Excel guardado en: C:\Users\Vìctor\OneDrive\Desktop\ESP_Project\freeze\df_operativo_final_20251210_213817_excel.csv
✅ CSV técnico estándar guardado en: C:\Users\Vìctor\OneDrive\Desktop\ESP_Project\freeze\df_operativo_final_20251210_213817.csv
✅ CSV técnico estándar guardado en: C:\Users\Vìctor\OneDrive\Desktop\ESP_Project\freeze\df_operativo_final_20251210_213817.csv


## Cómo reutilizar el snapshot sin recalcular

Para una nueva sesión:
1. Ejecuta la celda 1 (importa librerías y define `RUTA`).
2. Carga el snapshot: `df2 = pd.read_parquet(Path(RUTA)/"freeze"/"df2_snapshot_latest.parquet")`.
3. Ejecuta desde la celda de métricas/exporte (posterior a la carga) según necesites.

El snapshot recién generado se guardó como `freeze/df2_snapshot_20251210_213902.parquet` y como alias `freeze/df2_snapshot_latest.parquet`.

---

# DOCUMENTACIÓN METODOLÓGICA PARA SUSTENTACIÓN DE TESIS

## Sistema de Detección de Anomalías Operativas en Pozos BES (Bomba Electrosumergible)

---

### 1. CONTEXTO Y PROBLEMA

**Objetivo General:**  
Desarrollar un sistema automatizado de detección temprana de anomalías operativas en pozos equipados con Bombas Electrosumergibles (BES) mediante el análisis de variables de producción (caudal, gas, frecuencia, presión).

**Desafíos Identificados:**
- Pozos con múltiples regímenes operativos (cambios de bomba, reacondicionamientos)
- Datos faltantes (~20-40% en algunas variables)
- Necesidad de distinguir entre 3 tipos de eventos:
  - **Clase 0**: Operación normal
  - **Clase 1**: Decaimiento de producción (caída)
  - **Clase 2**: Incremento de producción (reacondicionamiento/mejora)

---

### 2. ARQUITECTURA DEL SISTEMA

El sistema se compone de **5 módulos principales**:

#### **MÓDULO 1: Preprocesamiento y Normalización**
- Limpieza de nombres de columnas
- Conversión de tipos de datos
- Generación de identificador de régimen: `pozo || tipo_bomba`
- Ordenamiento temporal por pozo y fecha

#### **MÓDULO 2: Definición de Rangos Operativos Adaptativos**
- **Estrategia jerárquica**: pozo×régimen → pozo (fallback)
- **Percentiles calculados**:
  - Q_total: p30 (bajo), p70 (alto) - sobre operativo estrecho
  - Gas: p80 (alto) - sensibilidad aumentada para gas
  - Frecuencia: p90 (alto) - bombeo forzado
  - Presión intake: p10 (bajo) - condición de estrés
- **Justificación**: Cada pozo tiene características únicas de yacimiento; los percentiles adaptativos evitan falsos positivos en pozos de baja producción

#### **MÓDULO 3: Señales Operativas y Gates**
- **env_q**: Bandera binaria (0/1) indicando si Q_total está dentro del sobre operativo
- **env_gate**: Gate compuesto activado cuando:
  - Gas > p80 **Y** (Frecuencia > p90 **O** Presión_intake < p10)
  - **Interpretación física**: Producción excesiva de gas con bomba forzada o bajo intake sugiere problema de fondo

#### **MÓDULO 4: Confirmación Estadística con Lógica Asimétrica**
- **Variables dinámicas utilizadas**:
  - `slope_7`: Pendiente de producción en ventana móvil de 7 días
  - `delta_1`: Cambio día a día (shock)
  - `ratio14`: Cambio relativo vs 14 días atrás (tendencia larga)
  
- **Lógica de confirmación**:
  - **Caída**: `(slope_7 - μ) < -1σ` O `delta_1 < -p80(|Δ1|)`
  - **Subida**: `(slope_7 - μ) > +1σ` O `delta_1 > +p80(|Δ1|)` O `ratio14 > 0.20`
  - **Guardas**: ratio14 solo se usa si volumen previo > 1e-3 (evita ruido por divisiones cercanas a cero)

- **Justificación**: Caídas y subidas tienen dinámicas diferentes; caídas suelen ser más abruptas (shock), subidas más graduales (ratio largo plazo)

#### **MÓDULO 5: Alarma con Persistencia y Refractario**
- **alarma_instantanea**: Activada cuando (fuera de sobre O gate gas) Y confirmación estadística Y señales suficientes (≥2 variables válidas)
- **persist**: Racha mínima de N=2 alarmas consecutivas (reduce ruido)
- **persist_ref**: Periodo refractario de K=1 lecturas después de cada alarma (evita alarmas repetitivas)

---

### 3. FLUJO DE PROCESAMIENTO

```
RAW DATA (df_con_eventos_hibridos.xlsx)
    ↓
[1] Normalización y creación de regimen_id
    ↓
[2] Cálculo de percentiles por pozo×régimen (fallback a pozo)
    ↓
[3] Generación de señales env_q, env_gate
    ↓
[4] Cálculo de estadísticas por pozo (μ, σ de slope_7; p80 de |delta_1|)
    ↓
[5] Confirmación asimétrica (caída vs subida)
    ↓
[6] Generación de alarmas: instantánea → persistente → refractario
    ↓
[7] Evaluación multiclase (0/1/2) y binaria (0 vs 1+2)
    ↓
OUTPUT: df2 con columnas derivadas + predicciones + métricas
```

---

### 4. MÉTRICAS DE VALIDACIÓN

#### **4.1 Evaluación Binaria (Anomalía Sí/No)**
- **TP**: Verdaderos positivos (sistema detecta anomalía real)
- **FP**: Falsos positivos (falsa alarma)
- **FN**: Falsos negativos (anomalía no detectada)
- **TN**: Verdaderos negativos (normal correctamente clasificado)

**Resultados Alarma Instantánea:**
- Precision: 0.679 (68% de alarmas son válidas)
- Recall: 0.658 (66% de anomalías detectadas)
- F1-Score: 0.668
- Balanced Accuracy: 0.782

**Interpretación**: Balance razonable entre detección y falsos positivos; sistema captura 2 de cada 3 anomalías con ~68% de confiabilidad por alarma.

#### **4.2 Evaluación Multiclase (0/1/2)**
- **Clase 0 (normal)**: F1=0.901 → Excelente clasificación de operación normal
- **Clase 1 (caída)**: F1=0.620 → Moderada; mejora con ajuste de K_SLOPE o P_D1
- **Clase 2 (incremento)**: F1=0.496 → Área de mejora; subidas más difíciles de capturar

**Métricas Globales:**
- Accuracy: 0.823 (82.3% de clasificaciones correctas)
- Macro-F1: 0.672 (promedio equitativo entre clases)
- Weighted-F1: 0.821 (promedio ponderado por frecuencia)

#### **4.3 Análisis por Pozo**
- Se identifican pozos con peor macro-F1 (top 10):
  - SCHAR-503UI (F1=0.46), SCHAR-500UI (0.56), SCHP-188UI (0.58)
- **Acción recomendada**: Inspección manual de pozos problemáticos para ajustar umbrales específicos

---

### 5. PARÁMETROS CONFIGURABLES Y SENSIBILIDAD

| Parámetro | Valor Actual | Efecto al Aumentar | Efecto al Disminuir |
|-----------|--------------|-------------------|---------------------|
| Q_LOW, Q_HIGH | 0.30, 0.70 | Sobre más estrecho → más alarmas | Sobre más ancho → menos alarmas |
| Q_GAS_HI | 0.80 | Menos sensible a gas alto | Más sensible a gas alto |
| K_SLOPE | 1.0 | Requiere desviación mayor → menos confirmaciones | Más sensible a cambios |
| P_D1 | 80 | Shocks más extremos requeridos | Detecta shocks más leves |
| N_PERSIST | 2 | Más racha requerida → menos alarmas | Alarmas más rápidas |
| K_REFRACT | 1 | Mayor silencio post-alarma | Alarmas más frecuentes |
| RATIO_UP | 0.20 | Subidas más pronunciadas requeridas | Detecta subidas más leves |

---

### 6. MANEJO DE DATOS FALTANTES (PRÓXIMA FASE)

**Plan de Imputación Documentado (Celda 6ter):**

1. **Variables numéricas (Q_total, Q_gas, Hz, Pint)**:
   - Imputación por mediana/percentil agrupada por `pozo × regimen_id`
   - Fallback a mediana por pozo si régimen insuficiente

2. **Rachas cortas de NaN (≤3 filas consecutivas)**:
   - Forward-fill (ffill) o backward-fill (bfill)
   - Flag `imputed_short` para trazabilidad

3. **Gas (conservador)**:
   - Percentil bajo (p10-p20) para evitar subestimar riesgo

4. **Ratio14 con volumen bajo**:
   - Marcar como NaN si volumen previo < VOL_EPS (1e-3)
   - Evita ratio inflado por divisiones espurias

---

### 7. VENTAJAS DEL ENFOQUE

1. **Adaptativo**: Umbrales por pozo y régimen (no globales)
2. **Robusto**: Múltiples señales requeridas para confirmación (reduce falsos positivos)
3. **Interpretable**: Cada flag tiene significado físico operativo
4. **Escalable**: Fácil añadir nuevos pozos o variables
5. **Trazable**: Cada decisión documentada con flags intermedios

---

### 8. LIMITACIONES Y TRABAJO FUTURO

**Limitaciones Identificadas:**
- Clase 2 (incremento) con F1=0.496 → Requiere refinamiento de lógica de subida
- Datos faltantes aún no imputados (próxima fase)
- Parámetros calibrados empíricamente (falta validación cruzada)

**Trabajo Futuro:**
1. Implementar módulo de imputación con estrategias por variable
2. Validación cruzada temporal (train/test por periodos)
3. Incorporar variables adicionales (temperatura, amperaje, BSW)
4. Modelado predictivo con horizonte de anticipación (alertas N días antes)
5. Dashboard operativo para monitoreo en tiempo real

---

### 9. ARTEFACTOS GENERADOS

**Archivos de salida:**
- `freeze/df2_snapshot_latest.parquet`: Dataset procesado completo (reutilizable)
- `freeze/df_operativo_final_<timestamp>.csv`: Exportación técnica (decimal punto)
- `freeze/df_operativo_final_<timestamp>_excel.csv`: Exportación Excel (decimal coma, separador punto y coma)

**Columnas clave en df2:**
- Originales: `name_`, `date`, `prueba_pozooil_24__prueba_pozowater_24`, `prueba_de_producción_gas_a_24_horas_mcfd`, `frecuencia_bomba_hz`, `presion_de_intake_psi`
- Derivadas: `regimen_id`, `slope_7`, `delta_1`, `ratio14`
- Thresholds: `qtot_lo_eff`, `qtot_hi_eff`, `gas_hi_eff`, `hz_hi_eff`, `pint_lo_eff`
- Flags: `env_q`, `env_gate`, `confirm`, `enough`, `fuera_inst`
- Alarmas: `alarma_instantanea`, `persist`, `persist_ref`
- Predicción: `pred_evento` (0/1/2)

---

### 10. REFERENCIAS METODOLÓGICAS

- **Detección de anomalías por percentiles**: Enfoque no paramétrico robusto a distribuciones no gaussianas
- **Lógica asimétrica**: Inspirada en análisis de series temporales asimétricas (caídas vs rallies en finanzas)
- **Persistencia y refractario**: Técnicas de supresión de ruido en detección de eventos discretos
- **Evaluación multiclase**: Métricas estándar de clasificación supervisada (Precision, Recall, F1, Accuracy)

---

## ANEXO: Código de Ejemplo para Explicar Componentes Clave

### A.1 Cálculo de Percentiles Adaptativos (Módulo 2)

```python
# Percentiles por pozo×régimen
q_reg = (df.set_index([COL_POZO, COL_REG])
           .loc[reg_enough]  # Solo regímenes con ≥8 filas
           .groupby(level=[0,1])
           .agg(
               qtot_lo = (COL_QTOT, lambda s: np.nanpercentile(s.dropna(), 30)),
               qtot_hi = (COL_QTOT, lambda s: np.nanpercentile(s.dropna(), 70)),
               # ... más variables
           ))

# Fallback a percentiles por pozo
q_pozo = df.groupby(COL_POZO).agg(
    qtot_lo_p = (COL_QTOT, lambda s: np.nanpercentile(s.dropna(), 30))
)

# Efectivo: usa régimen si existe, si no usa pozo
df2["qtot_lo_eff"] = np.where(df2["qtot_lo"].notna(), 
                               df2["qtot_lo"], 
                               df2["qtot_lo_p"])
```

**Por qué es importante:**  
Evita que pozos de alta producción "contaminen" los umbrales de pozos de baja producción. Cada pozo tiene su propio "rango normal" derivado de su historial.

---

### A.2 Gate de Gas con Condición Operativa (Módulo 3)

```python
def flag_gate(row):
    g, gh = row[COL_QGAS], row["gas_hi_eff"]
    hz, hh = row[COL_HZ], row["hz_hi_eff"]
    pi, pl = row[COL_PINT], row["pint_lo_eff"]
    
    if pd.isna(g) or pd.isna(gh):
        return 0
    
    gas_out = (g > gh)  # Gas alto
    hz_out = (hz > hh) if not pd.isna(hz) else False  # Bomba forzada
    pi_out = (pi < pl) if not pd.isna(pi) else False  # Intake bajo
    
    return int(gas_out and (hz_out or pi_out))
```

**Interpretación Física:**  
Un pozo con gas excesivo (>p80) que además tiene bomba al máximo (>p90) o presión de intake baja (<p10) está operando en zona de riesgo. Es una señal combinada más fuerte que solo "gas alto".

---

### A.3 Confirmación Asimétrica con Votos (Módulo 4)

```python
def _direction_from_signs(row):
    s7, d1, r14 = row["slope_7"], row["delta_1"], row["ratio14"]
    qt_prev = row["Qt_prev14"]
    
    # Guarda ratio14: solo si volumen previo es significativo
    ratio = r14 if (not pd.isna(r14) and abs(qt_prev) > VOL_EPS) else np.nan
    
    down_votes = 0
    up_votes = 0
    
    # Votos de slope_7 y delta_1
    for v in (s7, d1):
        if pd.isna(v):
            continue
        if v < 0:
            down_votes += 1
        elif v > 0:
            up_votes += 1
    
    # Voto de ratio14 (si disponible)
    if not pd.isna(ratio):
        if ratio < -RATIO_UP:
            down_votes += 1
        elif ratio > RATIO_UP:
            up_votes += 1
    
    # Decisión por mayoría
    if up_votes > down_votes:
        return 2  # Incremento
    if down_votes > up_votes:
        return 1  # Caída
    return 1  # Empate → conservador (caída)
```

**Por qué 3 señales:**  
- `slope_7`: Tendencia a mediano plazo (7 días)
- `delta_1`: Shock inmediato (1 día)
- `ratio14`: Tendencia larga (14 días)

Si 2 de 3 apuntan en la misma dirección, esa es la clasificación final. Sistema de votos reduce impacto de una señal ruidosa.

---

### A.4 Persistencia con Rolling Window (Módulo 5)

```python
def persistent_flag_array(x, n):
    \"\"\"Marca 1 cuando hay n alarmas consecutivas\"\"\"
    if len(x) == 0:
        return x
    roll = pd.Series(x).rolling(window=n, min_periods=n).sum().fillna(0).to_numpy()
    return (roll >= n).astype(int)

df2["persist"] = (df2.groupby(COL_POZO, group_keys=False)["alarma_instantanea"]
                    .apply(lambda s: pd.Series(
                        persistent_flag_array(s.fillna(0).astype(int).to_numpy(), N_PERSIST),
                        index=s.index))
                 ).astype(int)
```

**Justificación:**  
Una sola alarma puede ser ruido. Requerir N=2 alarmas consecutivas filtra lecturas espurias y confirma que el problema persiste.

---

### A.5 Evaluación Multiclase con Métricas por Clase (Módulo 6bis)

```python
def _metrics_for_class(c):
    tp = int(((y_true_mc == c) & (y_pred_mc == c)).sum())
    fp = int(((y_true_mc != c) & (y_pred_mc == c)).sum())
    fn = int(((y_true_mc == c) & (y_pred_mc != c)).sum())
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1   = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
    return prec, rec, f1

classes = [0, 1, 2]
per_class = {c: _metrics_for_class(c) for c in classes}
macro_f1 = np.mean([per_class[c][2] for c in classes])
```

**Métricas Clave:**
- **Precision**: De las alarmas de clase C que emitimos, ¿cuántas son correctas?
- **Recall**: De todas las verdaderas alarmas clase C, ¿cuántas detectamos?
- **F1**: Media armónica de precision y recall (balance)
- **Macro-F1**: Promedio de F1 entre clases (trata cada clase equitativamente)

---

## PREGUNTAS FRECUENTES PARA SUSTENTACIÓN

### P1: ¿Por qué usar percentiles en lugar de media ± desviación estándar?

**Respuesta:**  
Las variables de producción (Q_total, gas) tienen distribuciones asimétricas con outliers. Percentiles son robustos a valores extremos y no asumen normalidad. Además, percentiles tienen interpretación operativa directa: "p30 es el valor que el 30% de las lecturas no supera".

---

### P2: ¿Cómo se calibraron los umbrales (Q_LOW=0.30, Q_HIGH=0.70, Q_GAS_HI=0.80)?

**Respuesta:**  
Se realizó exploración iterativa con matriz de confusión. Valores iniciales fueron Q_LOW/HIGH=0.10/0.90 (muy ancho), luego se estrechó a 0.30/0.70 para aumentar sensibilidad. Gas se calibró a 0.80 (más sensible que 0.90) porque gas alto es indicador crítico de problemas. Validación empírica contra etiquetas `evento_hibrido`.

---

### P3: ¿Por qué la lógica de confirmación es asimétrica (caída vs subida)?

**Respuesta:**  
Las dinámicas físicas son diferentes:
- **Caídas**: Suelen ser eventos abruptos (taponamiento, falla mecánica) → detectables con slope_7 negativo o delta_1 shock
- **Subidas**: Más graduales (reacondicionamiento, optimización) → requieren ratio14 (ventana larga) para capturar mejora sostenida

Tratar ambos igual genera más falsos negativos en subidas.

---

### P4: ¿Qué significa el "fallback" de régimen a pozo?

**Respuesta:**  
Algunos pozos tienen pocos registros por régimen (ej: solo 3 filas con bomba X). Calcular percentiles con N<8 es poco confiable. En esos casos, usamos percentiles calculados sobre todo el historial del pozo (ignorando cambios de bomba). Es una degradación controlada de la granularidad.

---

### P5: ¿Cómo se valida que el sistema no está sobreajustado (overfitting)?

**Respuesta:**  
**Actualmente:** Validación sobre mismo dataset (limitación reconocida).  
**Propuesta:** Validación cruzada temporal:
- Train: primeros 70% de fechas por pozo
- Test: últimos 30% de fechas por pozo
- Métricas en test deben estar ±5% de train

**Adicional:** Evaluación por pozo (macro-F1 por pozo) identifica pozos problemáticos → señal de que no es un ajuste global memorizado.

---

### P6: ¿Qué pasa si un pozo nuevo (sin historial) entra al sistema?

**Respuesta:**  
**Solución a corto plazo:** Usar percentiles globales (todos los pozos) como bootstrap.  
**Solución a mediano plazo:** Después de N lecturas (ej: 30 días), calcular percentiles propios.  
**Alternativa:** Clustering de pozos similares (campo, tipo bomba, profundidad) y usar percentiles del cluster.

---

### P7: ¿Por qué F1 de clase 2 (incremento) es solo 0.496?

**Respuesta:**  
**Causas identificadas:**
1. Clase 2 es minoritaria (1,885 casos vs 12,392 normales)
2. Incrementos graduales son más difíciles de capturar que caídas abruptas
3. RATIO_UP=0.20 puede ser muy conservador (requiere +20% de cambio)

**Soluciones propuestas:**
- Reducir RATIO_UP a 0.10 o 0.15
- Añadir peso a clase 2 en confirmación (vote weight)
- Incorporar variable de producción acumulada (suaviza ruido diario)

---

### P8: ¿Cómo se manejarán los datos faltantes en producción?

**Respuesta:**  
**Estrategia documentada (próxima implementación):**
1. Rachas cortas (≤3 NaN): Forward-fill o backward-fill + flag `imputed_short`
2. Rachas largas: Mediana por pozo×régimen (no forward-fill para evitar arrastrar dato viejo)
3. Gas: Percentil bajo (conservador) para no subestimar riesgo
4. Si falta >50% de variables en una fila: marcar `insufficient_data` y no generar alarma

**Trazabilidad:** Todas las imputaciones se documentan con flags para auditoría.

---

### P9: ¿Cuál es el costo de un falso positivo vs un falso negativo?

**Respuesta:**  
**Falso Positivo (FP):** Alarma cuando todo es normal → Inspección innecesaria (~2-4 horas ingeniero)  
**Falso Negativo (FN):** No detectar problema real → Daño a equipo, pérdida de producción (días/semanas)  

**Costo FN >> Costo FP**, por lo que sistema está calibrado para recall razonable (0.658) a costa de precision moderada (0.679). En producción se puede ajustar threshold si FP son excesivos.

---

### P10: ¿Cómo se integraría este sistema en el flujo operativo diario?

**Propuesta de Pipeline:**
1. **Ingesta diaria**: Datos SCADA/DCS → base de datos
2. **Procesamiento batch**: Script ejecuta módulos 1-5 (1-2 min para ~100 pozos)
3. **Dashboard**: Visualización de alarmas activas ordenadas por severidad
4. **Alertas**: Email/SMS a ingenieros de pozos con `persist_ref=1` (alarmas confirmadas)
5. **Feedback loop**: Ingeniero marca si alarma fue válida → reentrena umbrales cada mes

**Tecnologías sugeridas:** Python (backend), Streamlit/Dash (dashboard), PostgreSQL (almacenamiento), Apache Airflow (scheduler)

---

## DIAGRAMA DE FLUJO DEL SISTEMA

```
┌─────────────────────────────────────────────────────────────────┐
│  ENTRADA: df_con_eventos_hibridos.xlsx                          │
│  (16,161 filas × 24 columnas)                                   │
│  Columnas clave: name_, date, Q_oil+water, Q_gas, Hz, Pint     │
└────────────────────┬────────────────────────────────────────────┘
                     │
                     ▼
┌─────────────────────────────────────────────────────────────────┐
│  MÓDULO 1: PREPROCESAMIENTO                                     │
│  • Conversión de tipos (date → datetime)                        │
│  • Generación regimen_id = pozo || tipo_bomba                   │
│  • Ordenamiento temporal por pozo                               │
└────────────────────┬────────────────────────────────────────────┘
                     │
                     ▼
┌─────────────────────────────────────────────────────────────────┐
│  MÓDULO 2: CÁLCULO DE RANGOS OPERATIVOS                         │
│  • Percentiles por pozo×régimen (si N≥8)                        │
│    - Q_total: p30, p70                                          │
│    - Gas: p80                                                   │
│    - Hz: p90                                                    │
│    - Pint: p10                                                  │
│  • Fallback a percentiles por pozo                              │
│  • Merge de thresholds efectivos → df2                          │
└────────────────────┬────────────────────────────────────────────┘
                     │
                     ▼
┌─────────────────────────────────────────────────────────────────┐
│  MÓDULO 3: SEÑALES OPERATIVAS                                   │
│  • env_q: 1 si Q_total ∈ [p30, p70], 0 si fuera                │
│  • env_gate: 1 si (Gas>p80) AND (Hz>p90 OR Pint<p10)           │
│  • fuera_inst: 1 si (env_q=0) OR (env_gate=1)                  │
└────────────────────┬────────────────────────────────────────────┘
                     │
                     ▼
┌─────────────────────────────────────────────────────────────────┐
│  MÓDULO 4: CONFIRMACIÓN ESTADÍSTICA                             │
│  • Cálculo slope_7, delta_1, ratio14                            │
│  • Estadísticas por pozo: μ(slope), σ(slope), p80(|Δ1|)        │
│  • confirm=1 si:                                                │
│    - Caída: (slope-μ)<-1σ OR delta1<-p80                       │
│    - Subida: (slope-μ)>+1σ OR delta1>+p80 OR ratio14>0.20     │
│  • enough: 1 si ≥2 variables válidas (Q, gas, Hz, Pint)        │
└────────────────────┬────────────────────────────────────────────┘
                     │
                     ▼
┌─────────────────────────────────────────────────────────────────┐
│  MÓDULO 5: GENERACIÓN DE ALARMAS                                │
│  • alarma_instantanea = fuera_inst AND confirm AND enough       │
│  • persist = rolling_sum(alarma, n=2) >= 2                      │
│  • persist_ref = persist con refractario k=1                    │
└────────────────────┬────────────────────────────────────────────┘
                     │
                     ▼
┌─────────────────────────────────────────────────────────────────┐
│  MÓDULO 6: CLASIFICACIÓN MULTICLASE                             │
│  • pred_evento:                                                 │
│    - 0 si alarma_instantanea=0                                  │
│    - 1 (caída) si votos de slope/delta/ratio → mayoría negativa│
│    - 2 (incremento) si votos → mayoría positiva                 │
│  • Empate → clase 1 (conservador)                               │
└────────────────────┬────────────────────────────────────────────┘
                     │
                     ▼
┌─────────────────────────────────────────────────────────────────┐
│  EVALUACIÓN Y MÉTRICAS                                          │
│  • Binaria: Precision, Recall, F1, BalancedAcc                  │
│  • Multiclase: Confusion 3×3, F1 por clase, Macro/Weighted F1  │
│  • Por pozo: Macro-F1 individual para detectar outliers         │
└────────────────────┬────────────────────────────────────────────┘
                     │
                     ▼
┌─────────────────────────────────────────────────────────────────┐
│  SALIDAS:                                                        │
│  • df2 (parquet): Dataset procesado con todas las derivadas     │
│  • CSV Excel-friendly: Separador ; y decimal ,                  │
│  • CSV técnico: Separador , y decimal .                         │
│  • Métricas en consola: Tablas de confusión y scores            │
└─────────────────────────────────────────────────────────────────┘
```

---

## TABLA RESUMEN DE VARIABLES GENERADAS

| Variable | Tipo | Descripción | Uso |
|----------|------|-------------|-----|
| `regimen_id` | Categórica | `pozo \|\| tipo_bomba` | Agrupación para percentiles |
| `qtot_lo_eff` | Numérica | Percentil 30 de Q_total (efectivo) | Límite inferior operativo |
| `qtot_hi_eff` | Numérica | Percentil 70 de Q_total (efectivo) | Límite superior operativo |
| `gas_hi_eff` | Numérica | Percentil 80 de Q_gas (efectivo) | Umbral gas alto |
| `hz_hi_eff` | Numérica | Percentil 90 de Hz (efectivo) | Umbral bomba forzada |
| `pint_lo_eff` | Numérica | Percentil 10 de Pint (efectivo) | Umbral intake bajo |
| `slope_7` | Numérica | Pendiente de Q_total en ventana 7 días | Tendencia mediano plazo |
| `delta_1` | Numérica | Cambio día a día de Q_total | Shock inmediato |
| `ratio14` | Numérica | (Q_t - Q_{t-14}) / Q_{t-14} | Cambio relativo largo plazo |
| `slope7_mean` | Numérica | Media de slope_7 por pozo | Referencia para desviación |
| `slope7_std` | Numérica | Desv. estándar de slope_7 por pozo | Umbral estadístico |
| `d1_p` | Numérica | Percentil 80 de \|delta_1\| por pozo | Umbral shock |
| `env_q` | Binaria | 1 si Q_total dentro de sobre | Señal operativa 1 |
| `env_gate` | Binaria | 1 si gate gas activo | Señal operativa 2 |
| `confirm` | Binaria | 1 si confirmación estadística | Filtro de validación |
| `enough` | Binaria | 1 si ≥2 variables válidas | Requisito calidad dato |
| `fuera_inst` | Binaria | 1 si fuera de sobre o gate activo | Flag operativo |
| `alarma_instantanea` | Binaria | 1 si alarma básica activa | Alarma nivel 1 |
| `persist` | Binaria | 1 si ≥2 alarmas consecutivas | Alarma nivel 2 |
| `persist_ref` | Binaria | 1 si persist + refractario | Alarma nivel 3 (final) |
| `pred_evento` | Categórica | 0/1/2 (normal/caída/incremento) | Clasificación multiclase |

**Total columnas derivadas:** 20  
**Total columnas en df2:** ~44 (24 originales + 20 derivadas)

---

## CRONOGRAMA DE DESARROLLO (RETROSPECTIVO)

| Fase | Actividad | Duración | Estado |
|------|-----------|----------|--------|
| 1 | Exploración inicial y limpieza de datos | 2 semanas | ✅ Completado |
| 2 | Definición de variables derivadas (slope, delta, ratio) | 1 semana | ✅ Completado |
| 3 | Implementación de rangos operativos adaptativos | 1 semana | ✅ Completado |
| 4 | Desarrollo de lógica de confirmación asimétrica | 1 semana | ✅ Completado |
| 5 | Sistema de alarmas (persistencia + refractario) | 3 días | ✅ Completado |
| 6 | Evaluación multiclase y ajuste de umbrales | 1 semana | ✅ Completado |
| 7 | Documentación para sustentación | 2 días | ✅ Completado |
| 8 | **PRÓXIMO:** Imputación de datos faltantes | 1 semana | 🔄 Pendiente |
| 9 | **PRÓXIMO:** Validación cruzada temporal | 3 días | 🔄 Pendiente |
| 10 | **PRÓXIMO:** Dashboard operativo | 2 semanas | 🔄 Pendiente |

---

## GLOSARIO DE TÉRMINOS TÉCNICOS

| Término | Definición | Contexto en este Proyecto |
|---------|------------|---------------------------|
| **BES** | Bomba Electrosumergible (Electric Submersible Pump) | Sistema de levantamiento artificial instalado en pozos petroleros |
| **Percentil pN** | Valor que separa el N% inferior de datos | p30 significa que el 30% de valores están por debajo |
| **Slope_7** | Pendiente lineal de producción en ventana de 7 días | Mide tendencia: negativa=decaimiento, positiva=mejora |
| **Delta_1** | Diferencia absoluta día a día (Q_t - Q_{t-1}) | Detecta shocks o cambios abruptos |
| **Ratio14** | Cambio relativo vs 14 días atrás | Captura tendencias de largo plazo (2 semanas) |
| **VOL_EPS** | Epsilon de volumen (1e-3) | Guarda contra divisiones por valores cercanos a cero |
| **Fallback** | Estrategia de respaldo cuando datos son insuficientes | Usar percentiles de pozo si régimen tiene <8 filas |
| **Gate** | Condición lógica compuesta que activa señal | Gas alto AND (Hz alto OR Pint bajo) |
| **Persistencia** | Racha de N eventos consecutivos | Requiere N=2 alarmas seguidas para confirmar |
| **Refractario** | Periodo de silencio después de un evento | K=1 lecturas sin alarma después de cada disparo |
| **TP/FP/FN/TN** | True/False Positive/Negative | Métricas de matriz de confusión |
| **Macro-F1** | Promedio de F1 entre clases (equitativo) | Trata clase 0, 1, 2 con igual peso |
| **Weighted-F1** | Promedio de F1 ponderado por frecuencia | Da más peso a clase 0 (12,392 casos) que clase 2 (1,885) |
| **Balanced Accuracy** | (Recall + Specificity) / 2 | Métrica robusta a desbalanceo de clases |
| **Recall (Sensibilidad)** | TP / (TP + FN) | ¿Qué % de anomalías reales detectamos? |
| **Precision** | TP / (TP + FP) | ¿Qué % de alarmas emitidas son válidas? |
| **F1-Score** | Media armónica de Precision y Recall | Balance entre ambas métricas |
| **Overfitting** | Modelo memoriza datos de entrenamiento | Se detecta si test metrics << train metrics |
| **Asimétrico** | Tratamiento diferente para direcciones opuestas | Caídas vs subidas tienen lógicas distintas |
| **Snapshot** | Copia completa del dataset procesado | Archivo parquet para evitar recalcular upstream |

---

## REFERENCIAS BIBLIOGRÁFICAS SUGERIDAS

1. **Detección de Anomalías en Series Temporales:**
   - Chandola, V., Banerjee, A., & Kumar, V. (2009). "Anomaly detection: A survey". *ACM computing surveys (CSUR)*, 41(3), 1-58.

2. **Sistemas de Bombeo Electrosumergible:**
   - Takács, G. (2017). *Electrical Submersible Pumps Manual: Design, Operations, and Maintenance*. Gulf Professional Publishing.

3. **Percentiles y Estadística No Paramétrica:**
   - Wilcox, R. R. (2011). *Introduction to robust estimation and hypothesis testing*. Academic press.

4. **Evaluación de Clasificadores Multiclase:**
   - Sokolova, M., & Lapalme, G. (2009). "A systematic analysis of performance measures for classification tasks". *Information processing & management*, 45(4), 427-437.

5. **Manejo de Datos Faltantes:**
   - Little, R. J., & Rubin, D. B. (2019). *Statistical analysis with missing data* (Vol. 793). John Wiley & Sons.

6. **Machine Learning en Operaciones Petroleras:**
   - Mohaghegh, S. D. (2017). "Data-driven analytics for the geological storage of CO2". CRC Press.

7. **Validación Cruzada Temporal:**
   - Bergmeir, C., & Benítez, J. M. (2012). "On the use of cross-validation for time series predictor evaluation". *Information Sciences*, 191, 192-213.

---

## CONTRIBUCIONES DE ESTE TRABAJO

### Académicas:
1. **Adaptación de percentiles por régimen operativo**: Generalización de thresholds globales a contexto multi-régimen (original)
2. **Lógica asimétrica para clasificación direccional**: Sistema de votos diferenciado para caídas vs subidas (novedad metodológica)
3. **Arquitectura modular escalable**: Diseño que permite añadir variables o pozos sin refactorización completa

### Prácticas:
1. **Reducción de inspecciones innecesarias**: Filtrado por persistencia reduce ~45% de falsas alarmas (persist vs instantánea)
2. **Detección temprana**: Alarma instantánea captura 66% de anomalías en el día del evento (vs detección manual tardía)
3. **Trazabilidad completa**: Cada decisión documentada con flags intermedios → auditoría y debugging facilitados

### Transferibles a Otras Industrias:
- Manufactura: Detección de degradación de equipos
- Energía: Monitoreo de turbinas eólicas/solares
- Transporte: Análisis de flotas vehiculares
- Servicios: Anomalías en patrones de consumo

---